# Financial Transactions Anomaly Detection

This notebook covers the full workflow:
- Import libraries
- Create synthetic dataset with null and duplicate rows
- EDA before cleaning
- Data cleaning
- EDA after cleaning
- Basic plots
- Save cleaned data in `data` folder
- Feature selection
- Train/test split
- Train machine learning model
- Evaluate model
- Save model in `models` folder
- Reload model
- Predict using new transaction data


In [ ]:
# 1. Import libraries
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

In [ ]:
# 2. Create project folders
os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)

print('Folders created successfully!')

## 3. Create Synthetic Dataset

We are creating a beginner-friendly financial transactions dataset.  
It includes:
- normal transactions
- suspicious transactions
- null values
- duplicate rows


In [ ]:
# 3. Create synthetic dataset
np.random.seed(42)

n = 5000

transaction_ids = np.arange(100001, 100001 + n)
customer_ids = np.random.randint(1000, 2000, n)

transaction_type = np.random.choice(
    ['payment', 'transfer', 'cash_out', 'cash_in', 'purchase'], n
)

device_type = np.random.choice(
    ['mobile', 'web', 'ATM', 'POS'], n
)

location = np.random.choice(
    ['Chennai', 'Bangalore', 'Mumbai', 'Delhi', 'Hyderabad'], n
)

merchant_category = np.random.choice(
    ['grocery', 'electronics', 'travel', 'fashion', 'utility'], n
)

hour_of_day = np.random.randint(0, 24, n)

# Create transaction amounts
transaction_amount = np.round(np.random.normal(loc=5000, scale=2500, size=n), 2)
transaction_amount = np.where(transaction_amount < 100, 100, transaction_amount)

# Create account balances
account_balance_before = np.round(np.random.normal(loc=50000, scale=15000, size=n), 2)
account_balance_before = np.where(account_balance_before < 1000, 1000, account_balance_before)

balance_after_transaction = account_balance_before - transaction_amount
balance_after_transaction = np.where(balance_after_transaction < 0, 0, balance_after_transaction)

transactions_last_24h = np.random.randint(1, 12, n)
is_international = np.random.choice([0, 1], n, p=[0.85, 0.15])

# Create anomaly label using business-like conditions
is_anomaly = (
    ((transaction_amount > 15000) & (transactions_last_24h > 8)) |
    ((hour_of_day >= 0) & (hour_of_day <= 4) & (transaction_amount > 12000)) |
    ((is_international == 1) & (transaction_amount > 10000)) |
    ((balance_after_transaction < 500) & (transaction_amount > 12000))
).astype(int)

df = pd.DataFrame({
    'transaction_id': transaction_ids,
    'customer_id': customer_ids,
    'transaction_type': transaction_type,
    'device_type': device_type,
    'location': location,
    'merchant_category': merchant_category,
    'hour_of_day': hour_of_day,
    'transaction_amount': transaction_amount,
    'account_balance_before': account_balance_before,
    'balance_after_transaction': balance_after_transaction,
    'transactions_last_24h': transactions_last_24h,
    'is_international': is_international,
    'is_anomaly': is_anomaly
})

print('Dataset created successfully!')
print('Shape:', df.shape)
df.head()

In [ ]:
# 4. Add null values manually
df.loc[np.random.choice(df.index, 50, replace=False), 'device_type'] = np.nan
df.loc[np.random.choice(df.index, 50, replace=False), 'location'] = np.nan
df.loc[np.random.choice(df.index, 50, replace=False), 'transaction_amount'] = np.nan

# Add duplicate rows manually
duplicate_rows = df.sample(20, random_state=42)
df = pd.concat([df, duplicate_rows], ignore_index=True)

print('Null values and duplicate rows added successfully!')
print('Updated shape:', df.shape)

In [ ]:
# 5. Save raw dataset
df.to_csv('data/raw_transactions.csv', index=False)
print('Raw dataset saved to data/raw_transactions.csv')

## 4. EDA Before Cleaning

In [ ]:
# View first few rows
df.head()

In [ ]:
# Check dataset information
df.info()

In [ ]:
# Check missing values
df.isnull().sum()

In [ ]:
# Check duplicate rows
df.duplicated().sum()

In [ ]:
# Statistical summary
df.describe(include='all')

In [ ]:
# Class distribution
df['is_anomaly'].value_counts()

## 5. Basic Plots Before Cleaning

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='is_anomaly', data=df)
plt.title('Target Distribution Before Cleaning')
plt.show()

In [ ]:
plt.figure(figsize=(10, 5))
sns.histplot(df['transaction_amount'], bins=30, kde=True)
plt.title('Transaction Amount Distribution Before Cleaning')
plt.show()

## 6. Data Cleaning

In [ ]:
# Remove duplicate rows
df = df.drop_duplicates()

# Fill missing categorical values using mode
df['device_type'].fillna(df['device_type'].mode()[0], inplace=True)
df['location'].fillna(df['location'].mode()[0], inplace=True)

# Fill missing numerical values using median
df['transaction_amount'].fillna(df['transaction_amount'].median(), inplace=True)

print('Data cleaning completed successfully!')

In [ ]:
# Verify after cleaning
print('Missing values after cleaning:')
print(df.isnull().sum())

print('\nDuplicate rows after cleaning:', df.duplicated().sum())

In [ ]:
# Save cleaned dataset
df.to_csv('data/cleaned_transactions.csv', index=False)
print('Cleaned dataset saved to data/cleaned_transactions.csv')

## 7. EDA After Cleaning

In [ ]:
df.head()

In [ ]:
df.shape

## 8. Basic Plots After Cleaning

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='transaction_type', data=df)
plt.title('Transaction Type Distribution')
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(x='is_anomaly', y='transaction_amount', data=df)
plt.title('Transaction Amount vs Anomaly')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='device_type', hue='is_anomaly', data=df)
plt.title('Device Type vs Anomaly')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
sns.countplot(x='is_international', hue='is_anomaly', data=df)
plt.title('International Transactions vs Anomaly')
plt.show()

## 9. Encode Categorical Columns

In [ ]:
label_encoders = {}
categorical_cols = ['transaction_type', 'device_type', 'location', 'merchant_category']

for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

joblib.dump(label_encoders, 'models/label_encoders.pkl')
print('Categorical columns encoded and encoders saved successfully!')
df.head()

## 10. Correlation and Feature Selection

In [ ]:
plt.figure(figsize=(12, 8))
corr_matrix = df.corr(numeric_only=True)
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

In [ ]:
# Select useful features based on correlation and domain understanding
selected_features = [
    'transaction_type',
    'device_type',
    'location',
    'merchant_category',
    'hour_of_day',
    'transaction_amount',
    'account_balance_before',
    'balance_after_transaction',
    'transactions_last_24h',
    'is_international'
]

X = df[selected_features]
y = df['is_anomaly']

print('Selected features:')
print(selected_features)

## 11. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

## 12. Feature Scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'models/scaler.pkl')
print('Scaler saved successfully!')

## 13. Train Machine Learning Model

**Chosen algorithm: Random Forest Classifier**  
Reason:
- works well for binary classification
- handles nonlinear relationships
- easy to understand
- gives feature importance
- suitable for beginner projects


In [ ]:
model = RandomForestClassifier(
    n_estimators=150,
    max_depth=10,
    random_state=42,
    class_weight='balanced'
)

model.fit(X_train_scaled, y_train)
print('Model trained successfully!')

## 14. Model Evaluation

In [ ]:
y_pred = model.predict(X_test_scaled)

accuracy = accuracy_score(y_test, y_pred)
print('Accuracy:', round(accuracy, 4))

print('\nClassification Report:')
print(classification_report(y_test, y_pred))

print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred))

## 15. Feature Importance

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': selected_features,
    'Importance': model.feature_importances_
}).sort_values(by='Importance', ascending=False)

feature_importance

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance)
plt.title('Feature Importance')
plt.show()

## 16. Save Model

In [ ]:
joblib.dump(model, 'models/anomaly_model.pkl')
print('Model saved successfully in models/anomaly_model.pkl')

## 17. Reload Model

In [ ]:
loaded_model = joblib.load('models/anomaly_model.pkl')
loaded_scaler = joblib.load('models/scaler.pkl')
loaded_encoders = joblib.load('models/label_encoders.pkl')

print('Model, scaler, and encoders reloaded successfully!')

## 18. Prediction with New Dataset

In [ ]:
# Create new transaction data for testing prediction
new_data = pd.DataFrame({
    'transaction_type': [loaded_encoders['transaction_type'].transform(['transfer'])[0]],
    'device_type': [loaded_encoders['device_type'].transform(['mobile'])[0]],
    'location': [loaded_encoders['location'].transform(['Mumbai'])[0]],
    'merchant_category': [loaded_encoders['merchant_category'].transform(['electronics'])[0]],
    'hour_of_day': [2],
    'transaction_amount': [18000],
    'account_balance_before': [25000],
    'balance_after_transaction': [7000],
    'transactions_last_24h': [10],
    'is_international': [1]
})

new_data

In [ ]:
# Scale the new input data
new_data_scaled = loaded_scaler.transform(new_data)

# Predict
prediction = loaded_model.predict(new_data_scaled)
prediction_proba = loaded_model.predict_proba(new_data_scaled)

print('Prediction:', prediction[0])
print('0 = Normal Transaction, 1 = Anomaly Transaction')
print('Prediction Probability:', prediction_proba)

## 19. Final Conclusion

In this notebook, we:
- created a synthetic financial transaction dataset
- added null and duplicate values
- performed EDA before and after cleaning
- cleaned the data
- selected important features
- trained a Random Forest model
- evaluated the model
- saved and reloaded the model
- predicted anomaly for a new transaction
